# Amaliyot 15. LDA mavzuli modellashtirish

15-interaktiv laboratoriyaga («Mavzuli modellashtirish») hamroh. Kurs mavzusi: latent Dirixle taqsimoti va semantik guruhlash

Daftar oʻz-oʻzidan yetarli: birinchi katak `common.py` faylini oʻquv korpusi va yordamchi funksiyalar bilan yaratadi. Kataklarni yuqoridan pastga ishga tushiring. Korpus oʻquv (shablon) boʻlib, haqiqiy masalada oʻz CSV faylingizni quyidagi ustunlar bilan qoʻying: `text` va `label`.

In [ ]:
%%writefile common.py
"""Matnlarni tasniflash va klasterlash amaliyotlari uchun umumiy funksiyalar."""
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams["figure.dpi"] = 110
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.25

TOPICS = {
    "sport": ("oʻyin jamoa gol oʻyinchi murabbiy chempionat turnir gʻalaba magʻlubiyat mavsum stadion muxlis hakam hisob "
              "himoyachi hujumchi darvozabon kubok liga final transfer klub zarba penalti musobaqa sportchi "
              "rekord medal olimpiada mashq durang").split(),
    "siyosat": ("qonun deputat saylov partiya ovoz prezident vazir hukumat parlament islohot byudjet "
                "sanksiya nomzod koalitsiya vazirlik farmon tuzatish loyiha muxolifat referendum diplomatiya "
                "hokim konstitutsiya majlis komissiya muzokara sammit saylovchi mandat vakolat").split(),
    "oshxona": ("shoʻrva retsept tuz goʻsht pishirmoq duxovka sabzavot tova qozon xamir shakar ziravor qaynatmoq qovurmoq "
                "dimlamoq sous bulyon salat oshpaz taom masalliq yogʻ un tuxum sut sarimsoq piyoz garmdori "
                "kartoshka shirinlik pishiriq marinad").split(),
    "texnologiya": ("protsessor dastur ilova smartfon neyrotarmoq algoritm maʼlumot server yangilanish dasturchi "
                    "kompyuter xotira ekran kamera brauzer tarmoq internet bulut kod platforma foydalanuvchi qurilma "
                    "chip batareya noutbuk dasturiy videokarta tizim interfeys shifrlash").split(),
}
NEUTRAL = ("bugun juda yangi aytishadi keyin ham mumkin edi boʻldi maʼlum qilindi kecha yaqinda hali hammasi "
           "bir nechta muhim birinchi").split()


def make_corpus(n_per_class=150, noise=0.3, length=(6, 11), seed=0):
    """4 mavzudagi qisqa «hujjatlar»dan iborat oʻquv korpusi.
    noise: oʻz mavzusidan boʻlmagan soʻzlar ulushi (neytral yoki begona). noise qancha katta boʻlsa, masala shuncha qiyin."""
    rng = random.Random(seed)
    names = list(TOPICS)
    rows = []
    for label in names:
        others = [t for t in names if t != label]
        for _ in range(n_per_class):
            words = []
            for _ in range(rng.randint(*length)):
                u = rng.random()
                if u < noise * 0.5:
                    words.append(rng.choice(NEUTRAL))
                elif u < noise:
                    words.append(rng.choice(TOPICS[rng.choice(others)]))
                else:
                    words.append(rng.choice(TOPICS[label]))
            rows.append((" ".join(words), label))
    df = pd.DataFrame(rows, columns=["text", "label"])
    return df.sample(frac=1, random_state=seed).reset_index(drop=True)


def project_2d(X, seed=0):
    """TF-IDF siyrak matritsasini tekislikka proyeksiyalash (TruncatedSVD)."""
    from sklearn.decomposition import TruncatedSVD
    return TruncatedSVD(2, random_state=seed).fit_transform(X)


def scatter2d(Z, labels, title="", ax=None):
    ax = ax or plt.gca()
    codes, uniq = pd.factorize(pd.Series(labels))
    sc = ax.scatter(Z[:, 0], Z[:, 1], c=codes, cmap="tab10", s=14, alpha=0.8)
    ax.set_title(title)
    handles = [plt.Line2D([], [], marker="o", ls="", color=sc.cmap(sc.norm(i)), label=str(u)) for i, u in enumerate(uniq)]
    ax.legend(handles=handles, fontsize=8)
    return ax


def top_terms(coef, terms, class_names, n=8):
    """Har bir sinf uchun eng katta vaznli soʻzlarni chiqaradi (chiziqli modellar)."""
    terms = np.asarray(terms)
    for i, name in enumerate(class_names):
        idx = np.argsort(coef[i])[::-1][:n]
        print(f"{name:12s}", ", ".join(terms[idx]))


In [ ]:
from common import *

# Oʻquv korpusi: 4 ta mavzu. noise — "begona" soʻzlar ulushi (qancha koʻp boʻlsa, shuncha qiyin).
df = make_corpus(n_per_class=150, noise=0.35, seed=0)
# Oʻquv korpusi oʻrniga oʻzingiznikini qoʻying: df = pd.read_csv("texts.csv")  # ustunlar: text, label
print(df.label.value_counts().to_dict())
df.head()

LDA **soʻzlar xaltasi** bilan ishlaydi (sanoqlar, TF-IDF emas).

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation
from sklearn.model_selection import train_test_split

cv = CountVectorizer(min_df=3)
X = cv.fit_transform(df.text); terms = np.array(cv.get_feature_names_out())
lda = LatentDirichletAllocation(n_components=4, learning_method="batch", random_state=0).fit(X)
def show_topics(model, n=8):
    for i, comp in enumerate(model.components_):
        print(f"Mavzu {i}:", ", ".join(terms[comp.argsort()[::-1][:n]]))
show_topics(lda)

## 1. Hujjatlarda mavzular ulushi

In [ ]:
theta = lda.transform(X)
order = df.label.argsort().values
plt.figure(figsize=(8, 5)); plt.imshow(theta[order], aspect="auto", cmap="viridis"); plt.colorbar(label="mavzu ulushi")
plt.xlabel("mavzu"); plt.ylabel("hujjatlar (haqiqiy mavzu boʻyicha tartiblangan)"); plt.grid(False); plt.title("Hujjat-mavzu (θ)"); plt.show()
print(pd.crosstab(df.label, theta.argmax(1)))

## 2. Aralash hujjatlar: LDA bir nechta mavzuni koʻrsatadi

In [ ]:
mixed = ["oʻyin jamoa gol oʻyinchi qonun deputat saylov partiya", "shoʻrva retsept tuz goʻsht protsessor dastur algoritm maʼlumot"]
print(pd.DataFrame(lda.transform(cv.transform(mixed)).round(2), index=["sport+siyosat", "oshxona+texnologiya"]))

## 3. Mavzular sonini tanlash

In [ ]:
Xtr, Xte = train_test_split(X, test_size=0.3, random_state=0)
ks = [2, 3, 4, 5, 6, 8]; perp = [LatentDirichletAllocation(k, learning_method="batch", random_state=0).fit(Xtr).perplexity(Xte) for k in ks]
plt.plot(ks, perp, marker="o"); plt.xlabel("mavzular soni"); plt.ylabel("ajratilgan hujjatlarda perplexity"); plt.title("Kamroq yaxshiroq (lekin mavzularni koʻz bilan oʻqish kerak)"); plt.show()

Real ishda mavzularning interaktiv vizualizatsiyasi uchun `pyLDAvis`, yanada ilgʻor modellar uchun `gensim` va `BERTopic`.

## Topshiriqlar

1. Tanlang `doc_topic_prior` (α) va `topic_word_prior` (β). Hujjatlar va mavzular «tozaligi» qanday oʻzgaradi?
2. 6 ta mavzuli korpus yarating (oʻz soʻz roʻyxatlaringizni quyiga qoʻshing: `TOPICS`) va LDA 6 ta mavzuni topishini tekshiring.
3. Topilgan mavzularga insoniy nomlar bering va haqiqiy yorliqlar bilan solishtiring.